In [3]:
import os
import json
import pandas as pd
from tqdm import tqdm

DATA_PATH = r'C:\Users\Mubeen Khan\OneDrive\Desktop\UI Captioning\rico_dataset_v0.1_semantic_annotations\semantic_annotations'

images = [f for f in os.listdir(DATA_PATH) if f.lower().endswith('.png')]
jsons  = [f for f in os.listdir(DATA_PATH) if f.lower().endswith('.json')]

print(f'PNG  files: {len(images):,}')
print(f'JSON files: {len(jsons):,}')

PNG  files: 66,261
JSON files: 66,261


In [4]:
def extract_elements(node, elements, depth=0):
    if not isinstance(node, dict):
        return
    elements.append({
        'text'  : node.get('text', '').strip(),
        'label' : node.get('componentLabel', '').strip(),
        'bounds': node.get('bounds', []),
        'depth' : depth
    })
    for child in node.get('children', []):
        extract_elements(child, elements, depth + 1)

In [5]:
def generate_caption(elements):
    texts   = [e['text']  for e in elements if e['text']]
    labels  = [e['label'] for e in elements if e['label']]
    buttons = [e['text'] for e in elements if 'Button' in e['label'] and e['text']]
    inputs  = [e['text'] for e in elements if ('EditText' in e['label'] or 'Input' in e['label']) and e['text']]
    navs    = [e['text'] for e in elements if 'Navigation' in e['label'] and e['text']]
    label_set = set(labels)
    has_list  = any('List' in l for l in labels)
    has_image = any('Image' in l or 'Icon' in l for l in labels)

    if inputs:
        screen_type = 'form'
    elif navs:
        screen_type = 'navigation'
    elif has_list:
        screen_type = 'list'
    elif has_image and not texts:
        screen_type = 'media'
    else:
        screen_type = 'general'

    display_texts = texts[:6]
    parts = []

    if display_texts:
        if screen_type == 'form':
            parts.append('A form screen with fields ' + ', '.join(display_texts))
        elif screen_type == 'navigation':
            parts.append('A navigation screen showing ' + ', '.join(display_texts))
        elif screen_type == 'list':
            parts.append('A list screen displaying ' + ', '.join(display_texts))
        elif screen_type == 'media':
            parts.append('A media screen showing ' + ', '.join(display_texts))
        else:
            parts.append('A screen displaying ' + ', '.join(display_texts))
    else:
        parts.append('A ' + screen_type + ' screen')

    if inputs:
        parts.append('with input fields for ' + ', '.join(inputs[:3]))

    unique_buttons = [b for b in buttons[:3] if b not in display_texts]
    if unique_buttons:
        parts.append('containing buttons ' + ', '.join(unique_buttons))

    if navs:
        parts.append('with navigation options ' + ', '.join(navs[:3]))

    return '. '.join(parts) + '.'

In [6]:
def extract_ui_text(elements, max_tokens=60):
    tokens = []
    for e in elements:
        if e['text']:
            tokens.extend(e['text'].lower().split())
    return ' '.join(tokens[:max_tokens])


def extract_layout(elements, max_tokens=50):
    labels = []
    for e in elements:
        if e['label']:
            label = e['label'].split('.')[-1].strip()
            if label:
                labels.append(label)
    return ' '.join(labels[:max_tokens])


# Quick test
test_elements = [
    {'text': 'Login',            'label': 'TextView', 'bounds': [], 'depth': 1},
    {'text': 'Password',         'label': 'EditText', 'bounds': [], 'depth': 1},
    {'text': 'Forgot Password?', 'label': 'TextView', 'bounds': [], 'depth': 2},
    {'text': 'Sign In',          'label': 'Button',   'bounds': [], 'depth': 1},
    {'text': 'Create Account',   'label': 'Button',   'bounds': [], 'depth': 1},
]
print('Caption :', generate_caption(test_elements))
print('UI Text :', extract_ui_text(test_elements))
print('Layout  :', extract_layout(test_elements))

Caption : A form screen with fields Login, Password, Forgot Password?, Sign In, Create Account. with input fields for Password.
UI Text : login password forgot password? sign in create account
Layout  : TextView EditText TextView Button Button


In [7]:
rows    = []
skipped = 0

for jf in tqdm(jsons, desc='Processing JSONs'):
    try:
        with open(os.path.join(DATA_PATH, jf), encoding='utf-8') as f:
            data = json.load(f)

        elements = []
        extract_elements(data, elements)

        if not elements:
            skipped += 1
            continue

        caption = generate_caption(elements)
        ui_text = extract_ui_text(elements)
        layout  = extract_layout(elements)

        if len(caption) < 10 or (not ui_text and not layout):
            skipped += 1
            continue

        image_file = jf.replace('.json', '.png')

        if not os.path.exists(os.path.join(DATA_PATH, image_file)):
            skipped += 1
            continue

        rows.append({
            'image'  : image_file,
            'caption': caption,
            'ui_text': ui_text,
            'layout' : layout
        })

    except Exception:
        skipped += 1
        continue

print(f'Processed : {len(rows):,}')
print(f'Skipped   : {skipped:,}')

Processing JSONs: 100%|██████████| 66261/66261 [16:09<00:00, 68.34it/s]

Processed : 65,538
Skipped   : 723


In [8]:
df = pd.DataFrame(rows)
df = df.dropna()
df = df[df['caption'].str.strip().str.len() > 10]
df = df[df['ui_text'].str.strip().str.len() > 0]
df = df.reset_index(drop=True)

print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
df.head(5)

Shape: (59676, 4)
Columns: ['image', 'caption', 'ui_text', 'layout']


,image,caption,ui_text,layout
0,10.png,"A list screen displaying Learn, Every second m...",learn every second month you stay in the 7 mon...,Toolbar Text Icon Image List Item Text List It...
1,100.png,"A screen displaying Log In, LOG IN WITH YOUR E...",log in log in with your email keep me logged i...,Toolbar Text Icon Icon Text Text Button Text T...
2,1000.png,"A screen displaying Pay bills, Quick resend.",pay bills quick resend,Image Text Button Text Button Image Text Image...
3,10000.png,"A screen displaying SNCF, TER, Les données son...",sncf ter les données sont actuellement fournie...,Icon Icon Text Text Text Text Image Text
4,10002.png,"A list screen displaying Mes favoris, Mes régi...",mes favoris mes régions mes alertes mes contac...,Icon Image Image Text Button Image Text Button...


In [9]:
from collections import Counter

first_bigrams = df['caption'].str.split().str[:2].str.join(' ').str.lower()
print('Top 10 opening phrases:')
for phrase, count in Counter(first_bigrams).most_common(10):
    print(f'  {count:,}x  "{phrase}"')

Top 10 opening phrases:
  33,609x  "a screen"
  23,135x  "a list"
  2,932x  "a form"


In [10]:
df.to_csv('final_fusion_dataset.csv', index=False)
df[['image', 'caption']].to_csv('final_generated_captions_full.csv', index=False)

print(f'fusion_dataset.csv saved — {len(df):,} rows')
print(f'generated_captions_full.csv saved — {len(df):,} rows')

fusion_dataset.csv saved — 59,676 rows
generated_captions_full.csv saved — 59,676 rows
